<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day18_practice3_%EB%AF%B8%EB%8B%88GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 미니 GPT - 다음 글자를 예측하는 (디코더)
# GPT-3 구조 동일
# "다음 글자 맞히기" 생성 = 예측 → 기존 단어에 붙이기 → 다시 예측의 반복

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, os, urllib.request
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# 셀 1. 데이터 - NSMC 리뷰를 이어붙인 '한국어 덩어리', document의 '글자'만 이어붙여 코퍼스로
URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
urllib.request.urlretrieve(URL, "ratings_train.txt")
print("ratings_train.txt 다운로드 완료")
reviews = []
with open("ratings_train.txt", encoding = "utf-8") as f:
  next(f) # 첫 줄(헤더 id document label)을 한 번 읽어 버림
  for line in f:
    parts = line.strip().split("\t")
    if len(parts) == 3 and len(parts[1]) > 10: # 본문이 10자 넘으면
      reviews.append(parts[1]) # 본문(parts[1])만 수집
    if len(reviews) >= 5000: break # 5000개 모으면 중단
text = " ".join(reviews) # 리뷰 5000개를 공백으로 이어 하나의 긴 글
print(f"코퍼스: 리뷰 {len(reviews):,}개, 총 {len(text):,}자")

ratings_train.txt 다운로드 완료
코퍼스: 리뷰 5,000개, 총 198,319자


In [12]:
chars = sorted(set(text)) # 고유 글자 목록
print(f"글자 사전 chars: {len(chars)}자 예:{chars}")
stoi = {c: i for i, c in enumerate(chars)} # dict {글자:번호}
itos = {i: c for c, i in stoi.items()} #dict {번호:글자}
data = torch.tensor([stoi[c] for c in text]) # (전체글자수,) 전체 텍스트를 번호 배열로
print(f"글자 번호 data: {data}")

글자 사전 chars: 1504자 예:[' ', '!', '"', '#', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '~', '·', '‥', '※', '→', '★', '♡', '♥', '、', '。', '『', '』', 'ㄱ', 'ㄴ', 'ㄵ', 'ㄷ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ', 'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ', 'ㅏ', 'ㅐ', 'ㅔ', 'ㅗ', 'ㅘ', 'ㅜ', 'ㅠ', 'ㅡ', 'ㅣ', '中', '乃', '人', '必', '故', '死', '甲', '要', '非', '가', '각', '간', '갈', '감', '갑', '값', '갓', '갔', '강', '갖', '갘', '같', '갚', '개', '객', '갠', '갱', '갸', '걍', '거', '걱', '건', '걷', '걸', '검', '겁', '것', '겄', '겉', '게', '겐', '겔', '겜', '겟', '겠', '겡', '겨', '격', '겪', '견', '결', '겸', '겹', '겻', '겼', '경', '곁', '곃', '계', '고', '곡', '곤', '곧', '골', '곰', '곱', '곳', '공', '과'

In [10]:
print(stoi)

{' ': 0, '!': 1, '"': 2, '#': 3, '%': 4, '&': 5, "'": 6, '(': 7, ')': 8, '*': 9, '+': 10, ',': 11, '-': 12, '.': 13, '/': 14, '0': 15, '1': 16, '2': 17, '3': 18, '4': 19, '5': 20, '6': 21, '7': 22, '8': 23, '9': 24, ':': 25, ';': 26, '<': 27, '=': 28, '>': 29, '?': 30, '@': 31, 'A': 32, 'B': 33, 'C': 34, 'D': 35, 'E': 36, 'F': 37, 'G': 38, 'H': 39, 'I': 40, 'J': 41, 'K': 42, 'L': 43, 'M': 44, 'N': 45, 'O': 46, 'P': 47, 'R': 48, 'S': 49, 'T': 50, 'U': 51, 'V': 52, 'W': 53, 'X': 54, 'Y': 55, 'Z': 56, '[': 57, ']': 58, '^': 59, '_': 60, '`': 61, 'a': 62, 'b': 63, 'c': 64, 'd': 65, 'e': 66, 'f': 67, 'g': 68, 'h': 69, 'i': 70, 'j': 71, 'k': 72, 'l': 73, 'm': 74, 'n': 75, 'o': 76, 'p': 77, 'r': 78, 's': 79, 't': 80, 'u': 81, 'v': 82, 'w': 83, 'x': 84, 'y': 85, 'z': 86, '~': 87, '·': 88, '‥': 89, '※': 90, '→': 91, '★': 92, '♡': 93, '♥': 94, '、': 95, '。': 96, '『': 97, '』': 98, 'ㄱ': 99, 'ㄴ': 100, 'ㄵ': 101, 'ㄷ': 102, 'ㄹ': 103, 'ㅁ': 104, 'ㅂ': 105, 'ㅃ': 106, 'ㅄ': 107, 'ㅅ': 108, 'ㅆ': 109, 'ㅇ': 110,

In [22]:
# 셀 2. 학습 데이터 만들기 - X와 y가 '한 칸 차이'
# 입력 x : 재 미 있
# 정답 y : 미 있 다 ← x를 한 칸 민 것
# 즉 매 위치에서 "다음 글자는?"을 동시에 풀게 한다.

SEQ = 64 # 시퀀스 모델이 한 번에 볼 수 있는 최대 글자 수, 컨텍스트 길이(context 맥락)
def get_batch(batch_size=64):
  ix = torch.randint(len(data) - SEQ - 1, (batch_size,)) # (batch_size) 시작 위치를 무작위로 batch_size개 뽑음
  x = torch.stack([data[i:i+SEQ] for i in ix]) # 각 시작점 i에서 SEQ 글자를 잘라 쌓음
  y = torch.stack([data[i+1:i+SEQ+1] for i in ix]) # 정답: 한 칸 뒤(다음 글자)
  return x.to(device), y.to(device) # (batch 64, SEQ 64)

x, y = get_batch(2)
print("입력 x[0] 앞 10자:", "".join(itos[int(i)] for i in x[0, :10]))
print("정답 y[0] 앞 10자:", "".join(itos[int(i)] for i in y[0, :10])) # ← 한 칸 밀림

입력 x[0] 앞 10자: 개 이병헌 마음같아
정답 y[0] 앞 10자:  이병헌 마음같아선


In [24]:
# 셀 3. 미니 GPT-3
class MultiHeadAttention(nn.Module):
  def __init__(self, dim, n_heads): # dim = , n_heads = 8 헤드 수
    super().__init__()
    assert dim % n_heads == 0 # 64/8 == 0
    self.n_heads = n_heads
    self.d_heads = dim // n_heads # d_head=8 헤드 하나가 맡는 차원 수

    self.W_q = nn.Linear(dim, dim, bias=False) # 각 헤드마다 Q,K,V를 만들 학습 행렬
    self.W_k = nn.Linear(dim, dim, bias=False)
    self.W_v = nn.Linear(dim, dim, bias=False)
    self.W_o = nn.Linear(dim, dim, bias=False) # 헤드들 결과를 합친 뒤 섞는 출력층

  def forward(self, x, mask=None):
    B, L, D = x.shape # x:(2,10,64) 배치, 단어수(Length), 벡터차원
    H, dh = self.n_heads, self.d_heads # H=8 헤드수, dh=8 각헤드 차원수

    # 1. 쪼개기 x:(2,10,64) → (2,10,8,8) → (2,8,10,8)
    Q = self.W_q(x).reshape(B, L, H, dh).transpose(1, 2) # (2,8,10,8) 배치, 멀티헤드수, 단어길이(10), 벡터차원수(8)
    K = self.W_k(x).reshape(B, L, H, dh).transpose(1, 2)
    V = self.W_v(x).reshape(B, L, H, dh).transpose(1, 2)

    # 2. 각자 보기 : 셀프어텐션
    scores = Q @ K.transpose(-2, -1) / math.sqrt(dh) # [(2,8,10,10)] (B, H, QL, KL) 관련도 점수
    if mask is not None:
      scores = scores.masked_fill(mask, -1e9) # masked_fill(조건, 값) 조건이 True인 위치를 값으로 채운다. (-10억 (-무한대))가 softmax를 통과하면 0이 된다.
    attn = F.softmax(scores, dim=-1) # softmax에서 dim=-1 축을 따라 정규화, (축은 그대로 남아있고, 그 축의 값들을 합=1로) (2,8,10Q,10K)
    self.attn_map = attn # 시각화용 저장 (2,8,10Q,10K)
    out = attn @ V # (2,8,10,8) B,H,L,dh

    # 3. 이어붙이기 : 헤드 8개(각 8차원)를 다시 원래 64차원 한 줄로 이어붙여 입력과 같은 모양 복원(2,10,64)
    out = out.transpose(1,2).reshape(B, L, D) # (2,8,10,8) → (2,10,8,8) → (2,10,64)

    # 4. 섞기 : 헤드 8개 결과를 어떻게 조합할지도 학습(W_o)
    return self.W_o(out) # (2,10,64)

class TransformerBlock(nn.Module):
  def __init__(self, dim=64, n_heads=8):
    super().__init__()
    self.attn = MultiHeadAttention(dim, n_heads) # 단어끼리 정보 교환
    self.norm1 = nn.LayerNorm(dim)
    self.ffn = nn.Sequential(
        nn.Linear(dim, dim * 4), nn.ReLU(), nn.Linear(dim * 4, dim)
    )
    self.norm2 = nn.LayerNorm(dim)

  def forward(self, x, mask=None): # x:(2,10,64)
    x = self.norm1(x + self.attn(x, mask)) # ＋× 잔차
    x = self.norm2(x + self.ffn(x))
    return x # (2,10,64) 입력과 똑같은 모양

class MiniGPT(nn.Module):
  def __init__(self, vocab, dim=128, n_heads=8, n_blocks=2):
    super().__init__()
    self.tok = nn.Embedding(vocab, dim)
    self.pos = nn.Embedding(SEQ, dim)
    self.blocks = nn.ModuleList(TransformerBlock(dim, n_heads) for _ in range(n_blocks))
    self.head = nn.Linear(dim, vocab)
    mask = torch.triu(torch.ones(SEQ, SEQ, dtype = torch.bool), diagonal=1)
    self.register_buffer("mask", mask)
  def forward(self, x):
    L = x.size(1)
    h = self.tok(x) + self.pos(torch.arange(L, device=x.device))
    for b in self.blocks:
      h = b(h, self.mask[:L, :L])
    return self.head(h)

model = MiniGPT(len(chars)).to(device)
print(f"미니 GPT: {sum(p.numel() for p in model.parameters())/1e6:.2f}M 파라미터")
# GPT-3는 175,000M - 구조는 같고 크기만 다르다

미니 GPT: 0.79M 파라미터


In [25]:
# 셀 4. 학습 - "다음 글자 맞히기"

In [26]:
# 셀 5. 생성 - 예측하고, 기존 문장 끝에 붙이고, 그걸로 다시 예측하고